# Our PR2 MJCF in the Multiverse apartment

This notebook uses `MJCFParser` for both inputs:

- `semantic_digital_twin/resources/mujoco_resources/pr2/pr2.xml`
- `semantic_digital_twin/resources/mujoco_resources/apartment/apartment.xml`

It only composes and launches the scene. No PyCRAM actions are executed. Gravity and contacts are enabled. The apartment contributes extensive collision geometry; our current PR2 MJCF has collision enabled only on the four fingertips.

In [ ]:
import os
from pathlib import Path
from xml.etree import ElementTree as ET

import mujoco

from physics_simulators.mujoco_simulator import MujocoSimulator
from semantic_digital_twin.adapters.mjcf import MJCFParser
from semantic_digital_twin.adapters.multi_sim import MujocoBuilder, MujocoSynchronizer
from semantic_digital_twin.world_description.connections import FixedConnection

def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'semantic_digital_twin/resources/mujoco_resources').is_dir():
            return candidate
    raise FileNotFoundError('Run this notebook from inside the repository.')

REPO_ROOT = find_repo_root()
MUJOCO_RESOURCES = REPO_ROOT / 'semantic_digital_twin/resources/mujoco_resources'
PR2_MJCF = (MUJOCO_RESOURCES / 'pr2/pr2.xml').resolve()
APARTMENT_MJCF = (MUJOCO_RESOURCES / 'apartment/apartment.xml').resolve()
GENERATED_SCENE = Path('/tmp/pr2_multiverse_apartment.xml')
HEADLESS = os.environ.get('MUJOCO_HEADLESS', '0') == '1'

if not APARTMENT_MJCF.is_file():
    raise FileNotFoundError(APARTMENT_MJCF)
print(f'PR2:       {PR2_MJCF}')
print(f'Apartment: {APARTMENT_MJCF}')
print(f'Output:    {GENERATED_SCENE}')
print(f'Headless:  {HEADLESS}')

## Parse and merge both MJCF worlds

In [ ]:
pr2_world = MJCFParser(str(PR2_MJCF)).parse()

apartment_parser = MJCFParser(str(APARTMENT_MJCF))
# Both parsers otherwise create a root body named 'world'.
apartment_parser.spec.worldbody.name = 'apartment_world'
apartment_world = apartment_parser.parse()

pr2_world.merge_world(
    apartment_world,
    root_connection=FixedConnection(
        parent=pr2_world.root, child=apartment_world.root
    ),
)
world = pr2_world
print(f'{len(world.bodies)} bodies')
print(f'{len(world.connections)} connections')
print(f'{len(world.actuators)} actuators')

## Optional initial PR2 base pose

These are the three planar joints defined in our `pr2.xml`. Leave them at zero for the first launch.

In [ ]:
PR2_X = 0.0
PR2_Y = 0.0
PR2_YAW = 0.0

world.state[world.get_degree_of_freedom_by_name('base_x_joint').id].position = PR2_X
world.state[world.get_degree_of_freedom_by_name('base_y_joint').id].position = PR2_Y
world.state[world.get_degree_of_freedom_by_name('base_yaw_joint').id].position = PR2_YAW
world.notify_state_change()

## Build one MJCF scene and add lighting

The builder subclass enables the same inertia balancing used by our PR2 MJCF. The XML post-processing adds the headlight, skybox, two directional lights, gravity, and an implicit integrator.

In [ ]:
class BalancedMujocoBuilder(MujocoBuilder):
    def _start_build(self, file_path):
        super()._start_build(file_path)
        self.spec.compiler.balanceinertia = True
        self.spec.compiler.boundmass = 1e-6
        self.spec.compiler.boundinertia = 1e-6

BalancedMujocoBuilder().build_world(world, str(GENERATED_SCENE))

tree = ET.parse(GENERATED_SCENE)
root = tree.getroot()
compiler = root.find('compiler')
compiler.set('balanceinertia', 'true')
compiler.set('boundmass', '0.000001')
compiler.set('boundinertia', '0.000001')

option = root.find('option')
if option is None:
    option = ET.Element('option')
    root.insert(1, option)
option.set('gravity', '0 0 -9.81')
option.set('integrator', 'implicitfast')

asset = root.find('asset')
visual = root.find('visual')
if visual is None:
    visual = ET.Element('visual')
    root.insert(list(root).index(asset), visual)
ET.SubElement(visual, 'headlight', diffuse='0.7 0.7 0.7', ambient='0.35 0.35 0.35', specular='0.1 0.1 0.1')
ET.SubElement(visual, 'rgba', haze='0.15 0.25 0.35 1')
ET.SubElement(visual, 'global', azimuth='120', elevation='-20')
ET.SubElement(asset, 'texture', name='scene_skybox', type='skybox', builtin='gradient', rgb1='0.3 0.5 0.7', rgb2='0 0 0', width='512', height='3072')

worldbody = root.find('worldbody')
ET.SubElement(worldbody, 'light', name='ceiling_light', pos='0 0 6', dir='0 0 -1', directional='true', diffuse='0.8 0.8 0.8', specular='0.2 0.2 0.2')
ET.SubElement(worldbody, 'light', name='fill_light', pos='2 -2 4', dir='-0.3 0.3 -1', directional='true', diffuse='0.45 0.45 0.45', specular='0.1 0.1 0.1', castshadow='false')

ET.indent(tree, space='  ')
tree.write(GENERATED_SCENE, encoding='utf-8', xml_declaration=True)
model = mujoco.MjModel.from_xml_path(str(GENERATED_SCENE))
print(f'Compiled: {model.nbody} bodies, {model.njnt} joints, {model.nu} actuators, {model.ngeom} geoms, {model.nlight} lights')

## Launch the viewer

This starts MuJoCo with gravity and contacts enabled. It does not execute robot actions.

In [ ]:
simulator = MujocoSimulator(
    file_path=str(GENERATED_SCENE),
    _headless=HEADLESS,
    _step_size=0.001,
    config={
        'gravity': True,
        'contact': True,
        'integrator': mujoco.mjtIntegrator.mjINT_IMPLICITFAST,
    },
)
synchronizer = MujocoSynchronizer(_world=world, simulator=simulator)
simulator.start()
print('PR2 apartment scene started.')

## Stop before rerunning

In [ ]:
if 'synchronizer' in globals():
    synchronizer.stop()
if 'simulator' in globals():
    simulator.stop()
print('Scene stopped.')